# Introduction to Lexical Semantic Change Detection
## Introduction
Author: Dominik Schlechtweg (University of Stuttgart)

Notebook introducing lexical semantic change detection concepts through concrete application of a full high-performing pipeline. We will

1. parse publicly available online corpora available for two different time periods,
2. sample usages for target words from the corpus vocabulary in both time periods,
3. infer distributional meaning representations for each usage,
4. infer semantic proximity labels between usages from the meaning representations,
5. represent semantic proximities in a weighted graph (Word Usage Graph) per target word,
6. infer sense clusters and corresponding sense frequency distributions on these graphs for both time periods,
7. measure change across time periods from sense frequency distributions or semantic proximity labels, and
8. visualize graphs and clusterings for manual inspection.

In the process, I will point out difficulties, open research questions, alternatives and important references for further reading. Accompanying slides can be found at <https://github.com/Garrafao/LSCDIntro>.

Most of the concepts used in this notebook are introduced in 

- Dominik Schlechtweg. 2023. Human and Computational Measurement of Lexical Semantic Change. PhD thesis.
University of Stuttgart, Stuttgart, Germany. Available at <http://dx.doi.org/10.18419/opus-12833>.

If you use the code from this notebook, please consider citing this source together with the reference for this notebook:

- Dominik Schlechtweg. 2026. Jupyter notebook introducing basic concepts of lexical semantic change detection. Available at <https://github.com/Garrafao/LSCDIntro>.

## Backend constraints
The notebook was designed to run on small computing infrastructure, i.e., 1 CPU with 2GB RAM, with Python 3.13.11. If your kernel dies you may have to reduce the batch size for embedding inference below. With this infrastructure the notebook should complete running within ca. 10 min. With larger infrastructure and/or GPU access the notebook should run much faster.

In [36]:
import sys
print(sys.version_info) # check the python version you are using

sys.version_info(major=3, minor=13, micro=11, releaselevel='final', serial=0)


# Data
If we want to detect changes in word usage of a population of speakers, we need some measurement pf this usage. A convenient measurement is digitized text produced by individuals from that speaker population. The larger and more diverse such measurements are, the better. For illustration, we will use the [Leipzig Corpora Collection](https://corpora.uni-leipzig.de/en) offering public digitized text corpora for many languages in different sizes over roughly the last 20 years. The English downloadable corpora versions can be found [here](https://wortschatz-leipzig.de/en/download/eng). We will use English News 2005 (1M) and English News 2020 (1M).

Advantages:
- easy to load and use
- available in parallel formats across languages and years
- stable genre

Disadvantages:
- very small
- sentence order is randomized
- no preprocessing
- long-term changes may not be reflected 

In [37]:
%%capture 
# Command above supresses the output of the cell, remove if running fails for debugging
# Install packages we need
!{sys.executable} -m pip install datasets==3.6.0 # version important as newer ones fail with corpus used below

In [38]:
# Download Leipzig corpora
from datasets import load_dataset # this package contains the corpora 

ds = load_dataset("imvladikon/leipzig_corpora_collection", "links", trust_remote_code=True) # Load all corpora 
name2index = {row['data_id']:i for i, row in enumerate(ds["train"])} # needed to get corpus by name

corpus_name1 = 'eng_news_2005_10K' # change name for different languages or periods, alternatives:
corpus_name2 = 'eng_news_2020_10K' # 'deu_news_2020_10K' (German), 'ron_news_2015_10K' (Romanian), 'pol_news_2020_10K' (Polish)
corpus1meta = ds['train'][name2index[corpus_name1]] # get corpus meta info
corpus2meta = ds['train'][name2index[corpus_name2]]

print(corpus1meta)
print(corpus2meta)

{'id': '217', 'data_id': 'eng_news_2005_10K', 'url': 'https://downloads.wortschatz-leipzig.de/corpora/eng_news_2005_10K.tar.gz', 'language': 'English', 'language_short': 'eng', 'year': '2005', 'size': '10K'}
{'id': '282', 'data_id': 'eng_news_2020_10K', 'url': 'https://downloads.wortschatz-leipzig.de/corpora/eng_news_2020_10K.tar.gz', 'language': 'English', 'language_short': 'eng', 'year': '2020', 'size': '10K'}


In [39]:
# Get and inspect corpus data
corpus1 = load_dataset("imvladikon/leipzig_corpora_collection", corpus1meta["data_id"], split="train")
corpus2 = load_dataset("imvladikon/leipzig_corpora_collection", corpus2meta["data_id"], split="train")
for row in corpus1["sentence"][:5]: # print first five sentences
    print(row)

I didn't know it was police housing," officers quoted Tsuchida as saying.
You would be a great client for Southern Indiana Homeownership's credit counseling but you are saying to yourself "Oh, we can pay that off."
He believes the 21st century will be the "century of biology" just as the 20th century was the century of IT.
They even call the civil rights organization a bit hypocritical.
But while VRE is not a threat to healthy individuals, its effect on the four HIV patients is potentially serious.


In [40]:
%%capture
# Install package for preprocessing
!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_sm

In [41]:
import spacy

nlp = spacy.load("en_core_web_sm") # load English preprocessing model
sentence1 = corpus1["sentence"][0] # first sentence in first corpus for illustration 
print('sentence1:', sentence1, '\n')
doc = nlp(sentence1) # parse sentence 

for token in doc: # print tags for each token in sentence
    print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_,
            token.shape_, token.is_alpha, token.is_stop)

sentence1: I didn't know it was police housing," officers quoted Tsuchida as saying. 

I I PRON PRP nsubj X True True
did do AUX VBD aux xxx True True
n't not PART RB neg x'x False True
know know VERB VB ccomp xxxx True False
it it PRON PRP nsubj xx True True
was be AUX VBD ccomp xxx True True
police police NOUN NN compound xxxx True False
housing housing NOUN NN attr xxxx True False
, , PUNCT , punct , False False
" " PUNCT '' punct " False False
officers officer NOUN NNS nsubj xxxx True False
quoted quote VERB VBD ROOT xxxx True False
Tsuchida Tsuchida PROPN NNP dobj Xxxxx True False
as as ADP IN prep xx True True
saying say VERB VBG pcomp xxxx True False
. . PUNCT . punct . False False


In [42]:
# Process full corpus
import os

of = 'outputs/'
if not os.path.exists(of): # Create output folder
    os.mkdir(of)
for name, sentences in [(of+"corpus1.txt", corpus1["sentence"]), (of+"corpus2.txt", corpus2["sentence"])]: # iterate over both corpora
    skipped = 0 # count skipped sentences 
    with open(name, "w", encoding="utf-8") as f:
        for i, sentence in enumerate(sentences): # iterate over sentences in corpus
            if i % 2000 == 0: # print progress every 2000 usages
                print("Processed:", i, "...")
            doc = nlp(sentence) # parse sentence 
            tokens = [(token.text, token.lemma_, token.pos_) for token in doc] # gather text, lemma, pos for all words in sentence
            text, lemma, pos = list(zip(*tokens)) # get separate lists
            line = " ".join(text) + "\t" + " ".join(lemma) + "\t" + " ".join(pos) + "\n" # join with space and tab for storage
            try: # catch faulty corpus lines
                assert line.count("\t")==2
                assert len(text) == len(lemma) == len(pos)
            except AssertionError:
                skipped += 1
                continue
            f.write(line) # export line
    print("Skipped for bad formatting:", skipped)

Processed: 0 ...
Processed: 2000 ...
Processed: 4000 ...
Processed: 6000 ...
Processed: 8000 ...
Skipped for bad formatting: 119
Processed: 0 ...
Processed: 2000 ...
Processed: 4000 ...
Processed: 6000 ...
Processed: 8000 ...
Skipped for bad formatting: 64


In [43]:
# Get vocabulary and frequencies
freqs = [] # to store word frequencies for each corpus
for filename in [of+"corpus1.txt", of+"corpus2.txt"]: # iterate over parsed corpora
    lemma_pos2freq = {} # to store word frequencies per corpus
    with open(filename) as fp: # open corpus file
        for i, line in enumerate(fp): # iterate over corpus lines
            if i % 2000 == 0:
                print("Processed:", i, "...")
            text, lemma, pos = line.split('\t') # get separate lists
            lemma = lemma.split(" ") # split at spaces into tokens
            pos = pos.split(" ")
            for l, p in zip(lemma, pos): # iterate over lemma-pos combos
                if not (l,p) in lemma_pos2freq: # increase count if found
                    lemma_pos2freq[(l,p)] = 1
                else:
                    lemma_pos2freq[(l,p)] += 1
    freqs.append(lemma_pos2freq) # store one dict per corpus

freqs1, freqs2 = freqs # get per-corpus freqs
freqs1 = dict(sorted(freqs1.items(), key=lambda item: item[1], reverse=True)) # sort lemma-pos pairs by frequency 
freqs2 = dict(sorted(freqs2.items(), key=lambda item: item[1], reverse=True)) 
print('len(freqs1)', len(freqs1))
print('len(freqs2)', len(freqs2))
freqs1_content = {(l, p):f for (l, p), f in freqs1.items() if p in ['NOUN', 'VERB', 'ADJ']} # filter for content words
freqs2_content = {(l, p):f for (l, p), f in freqs2.items() if p in ['NOUN', 'VERB', 'ADJ']}
print('len(freqs1_content)', len(freqs1_content))
print('len(freqs2_content)', len(freqs2_content))
words_top_freq1 = [item for i, item in enumerate(freqs1_content.items()) if 0 < i < 10] # get some example words for high freq range in corpus 1
words_mid_freq1 = [item for i, item in enumerate(freqs1_content.items()) if 5000 < i < 5010] # mid
words_low_freq1 = [item for i, item in enumerate(freqs1_content.items()) if 10771 < i < 10781] # low
print('words_top_freq1:', words_top_freq1)
print('words_mid_freq1:', words_mid_freq1)
print('words_low_freq1:', words_low_freq1)
print('Frequency of \"arm\" in corpus1:', freqs1['arm', 'NOUN'])
print('Frequency of \"arm\" in corpus2:', freqs2['arm', 'NOUN'])

Processed: 0 ...
Processed: 2000 ...
Processed: 4000 ...
Processed: 6000 ...
Processed: 8000 ...
Processed: 0 ...
Processed: 2000 ...
Processed: 4000 ...
Processed: 6000 ...
Processed: 8000 ...
len(freqs1) 23461
len(freqs2) 21357
len(freqs1_content) 10781
len(freqs2_content) 10710
words_top_freq1: [(('year', 'NOUN'), 702), (('have', 'VERB'), 685), (('time', 'NOUN'), 348), (('make', 'VERB'), 333), (('first', 'ADJ'), 322), (('go', 'VERB'), 316), (('take', 'VERB'), 306), (('get', 'VERB'), 300), (('last', 'ADJ'), 284)]
words_mid_freq1: [(('lagoon', 'NOUN'), 2), (('tune', 'NOUN'), 2), (('down', 'VERB'), 2), (('function', 'VERB'), 2), (('layoff', 'NOUN'), 2), (('courtyard', 'NOUN'), 2), (('oak', 'NOUN'), 2), (('lieu', 'NOUN'), 2), (('postponement', 'NOUN'), 2)]
words_low_freq1: [(('nightcap', 'NOUN'), 1), (('borderline', 'NOUN'), 1), (('submarine', 'NOUN'), 1), (('interaction', 'NOUN'), 1), (('reprieve', 'NOUN'), 1), (('palatable', 'ADJ'), 1), (('potluck', 'NOUN'), 1), (('salad', 'NOUN'), 1)

In [44]:
# Get usages
import numpy as np

targets = [('arm', 'NOUN')] # define target words from corpus vocabulary 
seed=15 # set seed for reproducibility, check 10, 15 or 19
''' Will be uncommented later to process randomly sampled targets
rng = np.random.default_rng(seed=seed) # initialize random number generator 
words_min_freq1 = [(l,p) for (l,p), f in freqs1_content.items() if 10 < f < 20] # get words with low number of usages
words_min_freq2 = [(l,p) for (l,p), f in freqs2_content.items() if 10 < f < 20]
words_min_freq12 = list(set(words_min_freq1) & set(words_min_freq2)) # get intersection between corpora
targets = rng.choice(words_min_freq12, size=20, replace=False).tolist() # randomly sample feasible number of words
targets = [(l, p) for l, p in targets] # minor change in data format
'''
print('targets:', targets) 
usages = [] # to store usage lists per corpus
for filename in [of+"corpus1.txt", of+"corpus2.txt"]: # iterate over corpora 
    lemma_pos2usages = {lp:[] for lp in targets} # to store usages per corpus
    with open(filename) as fp: # open corpus file 
        for i, line in enumerate(fp): # iterate over corpus lines
            if i % 2000 == 0:
                print("Processed:", i, "...")
            text, lemma, pos = line.split('\t')
            lemma = lemma.split(" ")
            pos = pos.split(" ")
            for j, (l,p) in enumerate(zip(lemma, pos)): # iterate over lemma-pos pairs in sentence 
                if not (l,p) in targets: # skip if not target
                    continue
                else: # store target usage
                    tokens = text.split(" ") # get all tokens in sentence 
                    start_id = len(" ".join(tokens[:j]))+1 if j > 0 else 0 # get start character index of target word
                    end_id = start_id + len(tokens[j]) # get end character index
                    lemma_pos2usages[(l,p)].append((text, (start_id, end_id))) # store usage as text with character ids
    usages.append(lemma_pos2usages) # store one dict per corpus

usages1, usages2 = usages # get per-corpus usages
print('Usages of first target in corpus1:', targets[0], '\n')
for text, (start, end) in usages1[targets[0]]: # print all usages of first target in first corpus 
    text = text[:start] + '**' + text[start:end] + '**' + text[end:] # mark target in usage text
    print(text, '\n')

targets: [('arm', 'NOUN')]
Processed: 0 ...
Processed: 2000 ...
Processed: 4000 ...
Processed: 6000 ...
Processed: 8000 ...
Processed: 0 ...
Processed: 2000 ...
Processed: 4000 ...
Processed: 6000 ...
Processed: 8000 ...
Usages of first target in corpus1: ('arm', 'NOUN') 

It was alleged the cycling great urinated in a glass of champagne his wife was drinking and on another occasion threatened to rip her **arms** off . 

You drive under that tunnel on 16th Street , and the hair on the back of your **arm** stands up . 

And Israeli forces carried out an air strike on a Palestinian **arms** cache in the Gaza Strip Wednesday night , a military spokesman said . 

She said the sentencing was affected by Hurstâ€ ™ s display of vindictiveness and strong - **arm** tactics even during the trial . 

Those are the types of **arms** commonly used by insurgents in their bid to topple Iraq 's first democratically elected administration in half a century , and drive out U.S. forces . 

In the middle 

### Notes

- **Representativeness**: The language measurements we use in this notebook for illustration are often not sufficient to detect long-term changes reliably. What we will detect, are mere differences between word usage as reflected in he paricular samples we have at hand. In order to conclude that a genuine *lexical semantic change* affecting the majority of the underlying speaker population has taken place, we would have to assure the *representativeness* of samples of the full language population. Small usage samples from particular genres or authors are likely not representative of the full population, i.e., don't reflect semantic changes accurately. Hence, we should strive for samples as *large* and *diverse* as possible. Text evidence should be supplemented with evidence from other *modalities* such as spontaneous speech data. The comparison of time *periods* pairs should be done for as many pairs as possible.
- **Rare senses**: New or almost lost senses are often *low* in usage *frequency*. This makes their detection a challenging needle-in-the-haystack problem. Their low frequency exacerbates the above-described representativeness problem as even larger and more diverse samples are needed to reliablly detect them.
- **Noise**: Standard parsers often do not detect multi-word expressions or idioms, or not reliably. Hence, e.g. usages of 'to back out' will be mixed into the usage samples as for 'to back so.' when treating ('back', 'VERB') as a target word. Similarly usages of 'fingers crossed' will be mixed together with usages of 'to cross sth.' when treating ('cross', 'VERB') as a target word. Also, parsers will make mistakes, especially for new, creative language use. The measurements (e.g. text) itself will also contain errors such as OCR errors or misspelled language.

### Open research questions

- How to define standard practices for corpus composition and usage sampling?
- How to determine sampling variables based on statistical significance and power?
- How to integrate high-quality preprocessing into semantic change detection pipelines?
- How to exploit corpus metadata like author or genre into pipelines?
- How to generally distinguish between differences observed due to sampling errors and "real" changes?
- How to combine evidence from different sources or time periods for change detection?
- How to detect low-frequency senses reliably and efficiently?

### Public diachronic text corpora

- [EN/DE/SV/LA corpora](https://www.ims.uni-stuttgart.de/en/research/resources/corpora/sem-eval-ulscd/) used in SemEval-2020 Task 1.
- [Deutsches Textarchiv](https://www.deutschestextarchiv.de/). Large German diachronic text corpus.
- EN/ES/DE/IT/NO/RU/SL/SV/LA/ZH [WUG datasets](https://www.ims.uni-stuttgart.de/data/wugs). DURel-style human-annotated usage samples for selected target words.

### Further reading
- Recent shared task on MWE detection: [Edition 2.0 of the PARSEME shared task on multilingual identification and paraphrasing of multiword expressions](https://aclanthology.org/2026.mwe-1.33/) (Scholivet et al., MWE 2026)
- Finding noisy usages: [Predicting Median, Disagreement and Noise Label in Ordinal Word-in-Context Data](https://aclanthology.org/2025.comedi-1.6/) (Choppa et al., CoMeDi 2025)
- Importance of corpus metadata: [The impact of lacking metadata for the measurement of cultural and linguistic change using the Google Ngram data sets—Reconstructing the composition of the German corpus in times of WWII](https://doi.org/10.1093/llc/fqv037) (Koplenig, DSH 2017)
- SemEval corpus creation: [SemEval-2020 Task 1: Unsupervised Lexical Semantic Change Detection](https://aclanthology.org/2020.semeval-1.1/) (Schlechtweg et al., SemEval 2020)
- Shared task on change discovery: [LSCDiscovery: A shared task on semantic change discovery and detection in Spanish](https://aclanthology.org/2022.lchange-1.16/) (Zamora-Reina et al., LChange 2022)
- Importance of preprocessing for BERT: [Explaining and Improving BERT Performance on Lexical Semantic Change Detection](https://aclanthology.org/2021.eacl-srw.25/) (Laicher et al., EACL 2021)
- Realistic detection of novel (often low frequency) senses: [APODICTUS: Automatic Processing Of DICTionary Update candidateS](https://garrafao.github.io/publications/251014-apodictus-slides.pdf) (Blessing et al., LREC 2026)

# Models

Standard LSCD models use a three-layer architecture:

1. semantic proximity (or WiC) labeling model,
2. clustering, and
3. change measurement.

## Semantic Proximity Model

We use the recent [XL-DURel](https://aclanthology.org/2025.findings-ijcnlp.19/) model with the cosine similarity, which was finetuned on most of the public [WUG datasets](https://www.ims.uni-stuttgart.de/data/wugs).

Advantages:

- high performance 
- small (<1B parameters) -> fast and deployable on small infrastructure 
- multilingual 
- optimized for ordinal semantic proximity labeling

Disadvantages:

- unclear how well it generalizes to new data/languages
- small size may inhibit performance in future
- needs additional inference step for thresholds
- pretrained mostly on modern text data

In [45]:
%%capture
# Install packages we need
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install xl_durel_utils
!{sys.executable} -m pip install networkx
!{sys.executable} -m pip install pyvis==0.1.9
!{sys.executable} -m pip install matplotlib

In [46]:
# Load model
from transformers import AutoTokenizer 
from xl_durel_utils.core import tokenize_truncate_decode
from sentence_transformers import SentenceTransformer, models
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("sachinn1/xl-durel") # load language model
tokenizer = AutoTokenizer.from_pretrained("sachinn1/xl-durel") # load its tokenizer

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [47]:
# Get embeddings 

# Gather all contexts for batch processing 
contexts = [] # to store all contexts 
i2tar2t = [] # to store mapping back to target and period
embeddings = {} # to store embeddings per target
i = 0
for (l, p) in targets: # iterare over targets
    embeddings[(l,p)] = [[],[]] # to store semantic representations (embeddings) per period
    for t, usages in enumerate([usages1[(l,p)], usages2[(l,p)]]): # iterare over usages per period
        if i < 5:
            print(l,p)
            print(' len(usages)', len(usages))
        for (text, indices) in usages: # iterate over usages
            context = tokenize_truncate_decode(text, indices, tokenizer, max_seq_len=128) # tokenize and mark target
            contexts.append(context) # store
            i2tar2t.append((i, (l,p), t)) # store info to map back later
            i+=1
            
batch_size = 128 # increase for faster processing with strong computing infrastructure 
embs = model.encode(contexts, batch_size=batch_size, show_progress_bar=True) # get embeddings of contexts 

for i, (l,p), t in i2tar2t: # map back embeddings to target and period
    embeddings[(l,p)][t].append(embs[i])

arm NOUN
 len(usages) 20


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [48]:
# Get proximities
similarities = {} # to store graded proximities
for (l,p) in targets: # iterate over targets
    embeddings1, embeddings2 = embeddings[(l,p)] # get embeddings per period
    sims = cosine_similarity(embeddings1+embeddings2) # get all pairwise proximities 
    similarities[(l,p)] = sims # store
    # Here we could map dense similarities to discrete ordinal values in [1,2,3,4]

print(similarities[targets[0]][:2]) # print all similarities for first two usages of first target

[[1.         0.86161363 0.27308705 0.38843045 0.2178539  0.80955786
  0.39106953 0.8523468  0.6913383  0.76964474 0.1445581  0.2175228
  0.8611997  0.16391718 0.8226919  0.85570407 0.8651049  0.6929728
  0.76452684 0.16992205 0.20577016 0.6478381  0.70655257 0.8513978
  0.2145484  0.17610647 0.6327801  0.49046987 0.8417809  0.8706248
  0.22797064 0.71668965 0.7903612  0.44090357 0.8608164 ]
 [0.86161363 1.0000002  0.24746826 0.45931184 0.19747749 0.7946532
  0.44782445 0.8889193  0.7809316  0.76232255 0.12764065 0.20492828
  0.84934115 0.15648337 0.8883903  0.96173596 0.917489   0.7836336
  0.83467436 0.16018835 0.20620856 0.6906948  0.79971194 0.83412373
  0.2113284  0.18292226 0.66753995 0.5149432  0.7988724  0.9448851
  0.21556482 0.7576896  0.85641474 0.48895946 0.9291414 ]]


### Graph Representation 

In [49]:
# Construct graph and visualize
import networkx as nx
from itertools import combinations 
from pyvis.network import Network
import numpy as np
import matplotlib.colors as mcolors
from textwrap import wrap
from sklearn.manifold import MDS
from IPython.display import display, HTML, IFrame

nice_colors = [x for x in mcolors.get_named_colors_mapping().values() if isinstance(x, str)] # Nice colors
colors_global = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#a65628', '#984ea3', '#999999', '#e41a1c', '#dede00'] # color-blind colors
colors_global = colors_global + nice_colors + ['#000000'] # concatenate 

def draw_graph(graph, clusters=None, skip=[], seed=None): 
    
    graph = graph.copy() # make sure not to modify the source graph
        
    if clusters is None: # if no clusters provided, assume all are in same cluster  
       clusters = {node:0 for node in graph.nodes()}

    G_int = Network(height='1000px', width='1200px', directed=False, notebook=True, bgcolor='#ffffff', font_color=False) # initialize interactive graph to plot
    
    weights = [graph[i][j]['weight'] for (i,j) in graph.edges()] # get all weights 

    # Find positions of nodes
    enan = [(u, v) for (u, v, d) in graph.edges(data=True) if np.isnan(d['weight'])] # get any potential invalid edges
    graph.remove_edges_from(enan)  # Remove nan edges for finding positions
    embedding = MDS(n_components=2, n_init=1, init="random", metric='precomputed', random_state=seed) # initialize MDS to map graph to 2D
    A = nx.adjacency_matrix(graph) # get the adjacency matrix of graph (similarities)
    dissimilarities = 1-A.toarray() # map to (cosine) distance as MDS takes dissimilarities as input 
    pos_matrix = embedding.fit_transform(dissimilarities) # map dissimilarities to 2D representation

    for node in graph.nodes(): # iterate over nodes
        if node in skip: # skip if told so
            continue
        x, y = pos_matrix[node]*1000 # get and scale 2D positions 
        label = node # label nodes with their id
        text, indices = graph.nodes[node]["usage"] # preprare usage for hover pop-up 
        id1, id2 = indices
        text = text[:id1] + '<b>' + text[id1:id2] + '</b>' + text[id2:]
        text = '<br>'.join(wrap(text, 70))
        G_int.add_node(n_id=node, label=str(label), shape='circle', size=1, physics=False, x=x, y=y, color=np.array(colors_global)[clusters[node]], title=text) # add node to graph

    for (i, j) in graph.edges(): # iterate over edges
        if i in skip or j in skip: # skip if either node is skipped
            continue
        weight = graph[i][j]['weight']
        label = round(weight, 2) # label edge with weight
        G_int.add_edge(i, j, color='lightgray', width=2, label=str(label), physics=False, weight=weight) # add edge to graph 
    
    return G_int # return graph object
    
graphs = {} # to store graphs
for (l, p) in targets: # iterate over targets
    usages = usages1[(l,p)]+usages2[(l,p)] # get usages
    u2i = {u:i for i, u in enumerate(usages)} # maps usage to index
    i2u = {i:u for i, u in enumerate(usages)} # maps index to usage   
    i2t = {i:1 if i < len(usages1[(l,p)]) else 2 for i, u in enumerate(usages)} # maps index to period
    pairs_id = list(combinations(i2u.keys(), 2)) # get all usage pairs as index pairs
    sim_matrix = similarities[(l,p)] # get all similarities 
    sims = [float(sim_matrix[i,j]) for i, j in pairs_id] # get similarities for all pairs

    graph = nx.Graph() # initialize graph
    graph.add_nodes_from(i2u.keys()) # add nodes with index label
    nx.set_node_attributes(graph, i2u, name="usage") # add usage as attribute 
    nx.set_node_attributes(graph, i2t, name="period") # add period as attribute 
    # Add edge data to graph
    edges_weighted = [(i, j, sims[k]) for k, (i, j) in enumerate(pairs_id)] # prepare similarities 
    graph.add_weighted_edges_from(edges_weighted) # ad similarities as weighted edges
    graphs[(l,p)] = graph # store final graph

    # Draw graph 
    graph_plot = draw_graph(graph, seed=seed) # get interactive graph plot
    graph_plot.save_graph(of+l+'-'+p+'.html') # save the plot to disk

In [50]:
# Visualize first target graph
# Could get stuck with larger graphs
l, p = targets[0]
display(IFrame(src=of+l+'-'+p+'.html', width=800, height=500))

## Graph Clustering Algorithm

We use the [Correlation Clustering](https://link.springer.com/article/10.1023/B:MACH.0000033116.57574.95) algorithm.

Advantages:

- finds number of clusters by itself
- handles missing edges
- robust to errors by minimizing a global loss
- optimizes an intuitive quality criterion
- has probabilistic interpretation
- controls granularity by threshold parameter
- is often used to cluster human annotations 

Disadvantages:

- cannot handle all distributios of clusters, e.g. assumes a global threshold
- heuristic, although it has a probabilistic interpretation
- huge search space, hence inefficient, but implementation is parallelizable

In [51]:
%%capture
# Install packages we need
!{sys.executable} -m pip install correlation-clustering

In [52]:
# Correlation Clustering 
from correlation_clustering.correlation import cluster_correlation_search

graphs_clustered = {} # to store clustered graphs
k = 0
for (l, p) in targets:
    graph = graphs[(l,p)].copy() # copy graph to keep original 
    
    # Prepare graph for clustering
    threshold = 0.483 # from XL-DURel paper appendix, change for different language
    for (i,j) in graph.edges():
        graph[i][j]['weight'] = graph[i][j]['weight']-threshold # shift edge weights by threshold

    # Cluster graph
    s = 5 # assume maximally 5 different senses per word    
    clusters, cluster_stats = cluster_correlation_search(graph, s = s) # cluster graph 

    # Display results
    node2cluster_inferred = {node:i for i, cluster in enumerate(clusters) for node in cluster} # mapping from nodes to cluster id
    node2cluster_inferred = {node:node2cluster_inferred[node] for node in graph.nodes()} # reorder
    if k < 5:
        print('clusters', node2cluster_inferred)
        print('loss', cluster_stats['loss'])

    nx.set_node_attributes(graph, node2cluster_inferred, name="cluster") # add cluster as attribute

    for (i,j) in graph.edges():
        graph[i][j]['weight'] = graph[i][j]['weight']+threshold # shift edge weights back    
    
    graphs_clustered[(l,p)] = graph # store graph with cluster info

    # Draw graph, this time with clusters 
    graph_plot = draw_graph(graph, clusters=node2cluster_inferred, seed=seed) # get interactive graph plot
    graph_plot.save_graph(of+l+'-'+p+'-clustered'+'.html') # save the plot to disk 
    k += 1

clusters {0: 0, 1: 0, 2: 1, 3: 0, 4: 1, 5: 0, 6: 3, 7: 0, 8: 0, 9: 0, 10: 2, 11: 4, 12: 0, 13: 2, 14: 0, 15: 0, 16: 0, 17: 0, 18: 0, 19: 2, 20: 2, 21: 0, 22: 0, 23: 0, 24: 1, 25: 1, 26: 0, 27: 0, 28: 0, 29: 0, 30: 1, 31: 0, 32: 0, 33: 0, 34: 0}
loss 1.7369304876327516


In [53]:
# Visualize first target graph
l, p = targets[0]
display(IFrame(src=of+l+'-'+p+'-clustered'+'.html', width=800, height=500))

## Change Measure

We derive time-wise sense frequency distributions from the clusterings and apply the standard ground-truth change measures [binary change, graded change](https://aclanthology.org/2020.semeval-1.1/) and [COMPARE](https://aclanthology.org/N18-2027/) to these.

In [54]:
# Get change scores
from scipy.spatial import distance

binary_change = {} # to store scores
graded_change = {}
COMPARE = {}
dists = {} # to store sense frequency distributions 
for (l, p) in targets: # iterate over targets
    graph = graphs_clustered[(l,p)] # get the clustered graph
    n2c = {node:graph.nodes()[node]['cluster'] for node in graph.nodes()} # get node to cluster mapping 
    n2t = {node:graph.nodes()[node]['period'] for node in graph.nodes()} # get node to period mapping 
    clusters = [{n for n, c_ in n2c.items() if c==c_} for c in set(n2c.values())] # get clusters as sets of nodes
    clusters.sort(key=lambda x:-len(x)) # sort by size
    
    # get sense frequency distributions
    clusters1 = [set([n for n in cluster if n2t[n]==1]) for cluster in clusters] # clusters per period
    clusters2 = [set([n for n in cluster if n2t[n]==2]) for cluster in clusters]
    distribution1 = [len(cluster) for cluster in clusters1] # distributions per period 
    distribution2 = [len(cluster) for cluster in clusters2]
    prob1 = list(distribution1/np.sum(distribution1)) # normalized distributions (probabilities)
    prob2 = list(distribution2/np.sum(distribution2))

    # measure change
    gain = int(np.sum([1 for (f1,f2) in zip(distribution1,distribution2) if f1==0 and f2>0])) # get gained clusters 
    loss = int(np.sum([1 for (f1,f2) in zip(distribution1,distribution2) if f2==0 and f1>0])) # get lost clusters 
    B = 1 if gain > 0 or loss > 0 else 0 # get binary change as gain or loss

    G = distance.jensenshannon(prob1, prob2, 2.0) # get graded change as JSD between distributions 

    wcompare=[d['weight'] for (u,v,d) in graph.edges(data=True) if (graph.nodes()[u]['period']==1 and graph.nodes()[v]['period']==2) or (graph.nodes()[u]['period']==2 and graph.nodes()[v]['period']==1)] # get edge weights from COMPARE group
    C = np.mean(wcompare) # get COMPARE score as mean of COMPARE edge weights

    binary_change[(l,p)] = B # store scores
    graded_change[(l,p)] = G
    COMPARE[(l,p)] = 1-C # invert to align in orientation with B and G
    dists[(l,p)] = [distribution1, distribution2] # store distributions 

    # Draw time-specific graphs
    for t in [1, 2]: # iterate over periods
        nodes_not_t = [n for n, t_ in n2t.items() if t_!=t] # get nodes not from that period to skip
        n2c_t = {n:c for n, c in n2c.items() if not n in nodes_not_t} # get node to cluster mapping 
        graph_plot = draw_graph(graph, clusters=n2c_t, skip=nodes_not_t, seed=seed) # plot interactive graph
        graph_plot.save_graph(of+l+'-'+p+'-clustered-'+str(t)+'.html') # save plot to disk 

In [55]:
# Visualize first target in period 1
l, p = targets[0]
display(IFrame(src=of+l+'-'+p+'-clustered-'+str(1)+'.html', width=800, height=500))
display(IFrame(src=of+l+'-'+p+'-clustered-'+str(2)+'.html', width=800, height=500))

In [56]:
# Show change scores and distributions 

binary_change = dict(sorted(binary_change.items(), key=lambda item: item[1], reverse=True)) # Reorder according to scores
graded_change = dict(sorted(graded_change.items(), key=lambda item: item[1], reverse=True)) 
COMPARE = dict(sorted(COMPARE.items(), key=lambda item: item[1], reverse=True)) 

print('binary_change', '\n', binary_change, '\n') # print scores
print('graded_change', '\n', graded_change, '\n')
print('COMPARE', '\n', COMPARE, '\n')
for (l,p) in targets:
    print('dists', (l,p), '\n', dists[(l,p)][0], '\n', dists[(l,p)][1], '\n')

binary_change 
 {('arm', 'NOUN'): 1} 

graded_change 
 {('arm', 'NOUN'): np.float64(0.275555327445624)} 

COMPARE 
 {('arm', 'NOUN'): np.float64(0.48284706483284634)} 

dists ('arm', 'NOUN') 
 [13, 2, 3, 1, 1] 
 [11, 3, 1, 0, 0] 



### Notes

- **LLMs**: [Recent studies](https://ebooks.iospress.nl/doi/10.3233/FAIA251313) do *not* suggest that LLMs yield higher performance for LSCD than smaller specialized models. However, systemic prompt engineering or finetuning could ultimately boost their performance.
- **Old language**: When using historical corpora as language usage measurements, orthographical, morphological or grammatical language changes put additional difficulties on models. Older word forms or sentence structures may be unfamiliar to the language model. To mitigate this, [some studies](https://arxiv.org/pdf/2010.03481) finetune the language model on historical corpora before applying them for usage encoding. This is however not commonly done.
- **Interpretability**: The clustering algorithm should be interpretable and validated in the sense to reflect cognitively and lexicographically adequate sense distinctions.
- **Ambiguity**: Correlation Clustering and many other clustering algorithms partition usages into discrete clusters. This ignores the modeling of word meaning uncertainty, i.e., ambiguity or vagueness.
- **Measures**: The current [SOTA for graded change prediction](https://arxiv.org/abs/2404.00176) skips clustering. From a modelling perspective, this is strange, as clustering is part of the ground-truth creation and should thus be modeled. The definition of better model measures of semantic change is an ongoing research topic, check e.g. [here](https://aclanthology.org/2026.lchange-1.13/) or [here](https://arxiv.org/html/2402.16596v1). 

### Open research questions 

- How can the potential of LLMs be made available for LSCD?
- How to make models applicable to old language corpora?
- How to model ground-truth clusterings more closely for hogher performance?
- Which clustering algorithms best reflect cognitively and lexicographically adequate sense distinctions?
- How to model ambiguity and vagueness of word meaning?
- Which aggregate measures of semantic change best approximate graded change?
- How to define and model binary change reliably?
definition generation 
- Ho to revisit [previous studies](https://aclanthology.org/P16-1141/) on statistical laws of change with modern models avoiding [previous fallacies](https://aclanthology.org/D17-1118/)?
- How to detect different [types of change](https://www.degruyterbrill.com/document/doi/10.1515/9783110804195.61/html)?
- What are the properties of word [sense frequency distributions](https://kilgarriff.co.uk/Publications/2004-K-TSD-CommonestSense.pdf)?


### Further reading 
- SOTA proximity model: [XL-DURel: Finetuning Sentence Transformers for Ordinal Word-in-Context Classification](https://aclanthology.org/2025.findings-ijcnlp.19/) (Yadav & Schlechtweg, Findings 2025)
- Proximity model comparison: [CoMeDi Shared Task: Median Judgment Classification & Mean Disagreement Ranking with Ordinal Word-in-Context Judgments](https://aclanthology.org/2025.comedi-1.4/) (Schlechtweg et al., CoMeDi 2025)
- Definition of Correlation Clustering: [Correlation Clustering](https://link.springer.com/article/10.1023/B:MACH.0000033116.57574.95) (Bansal et al., ML 2004)
- Comparison of clustering algorithms for LSCD: [Sense through time: diachronic word sense annotations for word sense induction and Lexical Semantic Change Detection](https://doi.org/10.1007/s10579-024-09771-7) (Schlechtweg et al., LRE 2024)
- Alternative interpretable clustering model: [Modeling Sense Structure in Word Usage Graphs with the Weighted Stochastic Block Model](https://aclanthology.org/2021.starsem-1.23/) (Schlechtweg et al., *SEM 2021)
- SOTA aggregate LSCD model comparison: [The LSCD Benchmark: a Testbed for Diachronic Word Meaning Tasks](https://arxiv.org/abs/2404.00176) (Schlechtweg et al., arXiv 2026)
- LLM evaluation: [Can Large Language Models compete with specialized models in Lexical Semantic Change Detection?](https://ebooks.iospress.nl/doi/10.3233/FAIA251313) (Zamora-Reina et al., ECAI 2025)
- Change score definition, annotation, clustering of ground-truth: [SemEval-2020 Task 1: Unsupervised Lexical Semantic Change Detection](https://aclanthology.org/2020.semeval-1.1/) (Schlechtweg et al., SemEval 2020)
- Change score definition and annotation: [Diachronic Usage Relatedness (DURel): A Framework for the Annotation of Lexical Semantic Change](https://aclanthology.org/N18-2027/) (Schlechtweg et al., NAACL 2018)
- Online annotation and prediction tool: [The DURel Annotation Tool: Human and Computational Measurement of Semantic Proximity, Sense Clusters and Semantic Change](https://aclanthology.org/2024.eacl-demo.15/) (Schlechtweg et al., EACL 2024)
- Model application for lexicography: [Automatic Non-recorded Sense Detection for Swedish through Word Sense Induction with fine-tuned Word-in-Context models](https://elex.link/elex2025/wp-content/uploads/eLex2025-11-Schlechtweg_etal.pdf) (Schlechtweg et al., eLex 2025)
- Generate sense definitions from clusters for interpretation: [Enriching Word Usage Graphs with Cluster Definitions](https://aclanthology.org/2024.lrec-main.546/) (Kutuzov et al., LREC-COLING 2024)

## Excercise 1

Choose a language you speak and try to find word sense differences in the corresponding Leipzig corpora. Adjust the code above to infer sense frequency distributions automatically and visualize the clustered graphs for inspection. Which words do have the highest and the lowest graded change score? Which senses do these words have and how do they differ across corpora? Find example usages. Present your findings to the class.

Variations: 
 - Try define a binary change score that is less sensitive to small clusters and use it for detection. Which words did you find and what are the gained and lost senses?
 - Try to find words with a high COMPARE score, but a rather low graded score, and vice versa. What do you observe? What can you say about their differences?

## Excercise 2

This notebook takes a primarily semasiological view, i.e., we start from words and check how they change. Think about how you could adjust the code above to take a more onomasiological view, i.e., start from senses and check how they are expressed. You do not need to literally "start" from senses. It's enough to show how a sense is expressed by different words. Choose an adequate example and adjust the code to visualize this example. Present it to the class.

## Exercise 3

Take a publicly available dictionary for some language you speak and choose a word of your interest. (You may use the example words from our slides.) Count the number of senses in the dictionary. Now, infer sense clusters for the word in a corresponding recent Leipzig corpus. Do the numbers align? Do the senses align? What are any sense differences? What could be reasons for these differences? Does clustering granularity play a role? Present your results to the class.

Example dictionaries: 
- [Oxford English Dictionary](https://www.oed.com/)
- [Swedish Academy Dictionary](https://svenska.se/)
- [Digitales Wörterbuch der deutschen Sprache](https://www.dwds.de/)

## Todos

- add usages for arm example from slides 
- plot change score distributions
- notebook for ground-truth validation with wug datasets
- notebook for WSD paradigm
- solve large graph visualization probe